Step 6- Analyzing and cleaning SYMPTOM_TEXT (truncating if no:tokens>512) using HuggingFace tokenizer (BioBERT’s wordpiece tokenizer)

In [1]:
import os
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer

# --- File paths (relative to project root) ---
SRC = os.path.join("..", "data", "processed", "sample_1k_with_severity.csv")
OUT = os.path.join("..", "data", "processed", "sample_1k_truncated_symptom_text.csv")

# --- Model/token limits ---
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
MAX_TOKENS = 512  # typical BERT/BioBERT max sequence length

# --- Load data ---
df = pd.read_csv(SRC, low_memory=False)

# --- Initialize tokenizer ---
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def truncate_to_max_tokens(text: str, max_tokens: int = MAX_TOKENS) -> str:
    """
    Truncate text to `max_tokens` using model tokenizer offsets.
    Returns the original text cut at the last fully included token.
    """
    if not isinstance(text, str) or not text.strip():
        return text  # keep as-is (NaN/empty)

    enc = tok(
        text,
        add_special_tokens=True,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_tokens
    )
    # Offsets include special tokens with (0,0). We want the last real token span end.
    offsets = enc["offset_mapping"]
    # Find the maximum 'end' offset among non-zero spans
    last_end = 0
    for (start, end) in offsets:
        if end > last_end:
            last_end = end

    # Safety clamp
    last_end = min(last_end, len(text))
    return text[:last_end]

# --- Missing rate ---
na_rate = df["SYMPTOM_TEXT"].isna().mean() * 100
print(f"\nSYMPTOM_TEXT missing: {na_rate:.2f}%")

# --- Precompute original lengths (optional diagnostics) ---
df["text_len_chars_before"] = df["SYMPTOM_TEXT"].astype(str).str.len()

# --- Token-based truncation ---
tqdm.pandas(desc="Truncating by tokens")
df["SYMPTOM_TEXT"] = df["SYMPTOM_TEXT"].progress_apply(lambda s: truncate_to_max_tokens(s, MAX_TOKENS))

# --- Post diagnostics ---
df["text_len_chars_after"] = df["SYMPTOM_TEXT"].astype(str).str.len()
truncated_rows = (df["text_len_chars_after"] < df["text_len_chars_before"]).sum()
print(f"✅ Token-based truncation applied with MAX_TOKENS={MAX_TOKENS}.")
print(f"ℹ️ Rows truncated: {truncated_rows}/{len(df)}")

# --- Optional: quick length stats ---
print("\n=== Length (chars) before → after (describe) ===")
print(pd.concat(
    [df["text_len_chars_before"].describe().rename("before"),
     df["text_len_chars_after"].describe().rename("after")],
    axis=1
))

# --- Save ---
df.to_csv(OUT, index=False)
print(f"\n💾 Saved to: {OUT}")


c:\Users\Mochitha vijayan\ADEGuardVAERS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



SYMPTOM_TEXT missing: 0.00%


Truncating by tokens: 100%|██████████| 1000/1000 [00:01<00:00, 692.60it/s]


✅ Token-based truncation applied with MAX_TOKENS=512.
ℹ️ Rows truncated: 65/1000

=== Length (chars) before → after (describe) ===
            before        after
count  1000.000000  1000.000000
mean    628.027000   549.922000
std     879.879686   602.178444
min       4.000000     4.000000
25%      94.750000    94.750000
50%     264.000000   264.000000
75%     899.250000   899.250000
max    8442.000000  2329.000000

💾 Saved to: ..\data\processed\sample_1k_truncated_symptom_text.csv
